# MiniMax-H3 Multi-GPU Benchmark Suite on 2x NVIDIA Tesla T4

**Objective:**
Empirically evaluate multi-GPU inference strategies on 2x NVIDIA Tesla T4 (16GB VRAM each, PCIe Gen3, No P2P):
1. **Module 1 (Text Encoder):** Truncated 50-layer Qwen3-VL-32B INT8 ConvRot (PP=2 vs TP=2 vs TP=2+SP+INT8 AG)
2. **Module 2 (Diffusion DiT):** MiniMax-H3 Ref2VA Pruned 50-block DiT INT8 ConvRot (PP=2 vs TP=2 vs TP=2+SP+INT8 AG)
3. **Module 3 (Video and Audio VAE Tiling):** Multi-GPU Spatial and Temporal Tile Decoding, Stereo Channel-Parallelism
4. **Module 4 (Video VAE INT8 ConvRot TP):** Full-frame ViT3D INT8 Decoder (Single GPU vs TP=2 FP16 AR vs TP=2+SP+INT8 AG)


In [ ]:
# Cell 1: Environment & Hardware Verification
import os, sys, time
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
num_gpus = torch.cuda.device_count()
print(f"Number of GPUs: {num_gpus}")

for i in range(num_gpus):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name} | VRAM: {props.total_memory / 1024**3:.2f} GB | SM: {props.major}.{props.minor}")

if num_gpus >= 2:
    can_p2p_01 = torch.cuda.can_device_access_peer(0, 1)
    can_p2p_10 = torch.cuda.can_device_access_peer(1, 0)
    print(f"\n[HARDWARE CHECK] GPU P2P Access (0->1): {can_p2p_01}, (1->0): {can_p2p_10}")
    if not can_p2p_01:
        print("-> CONFIRMED: 2x T4 does NOT support P2P. Setting NCCL_P2P_DISABLE=1.")
        os.environ["NCCL_P2P_DISABLE"] = "1"
        os.environ["NCCL_IB_DISABLE"] = "1"
    os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"


In [ ]:
# Cell 2: Install Dependencies & Setup Optimization Repo in /tmp
!pip install -q safetensors pandas matplotlib huggingface_hub comfy-kitchen comfy-aimdo einops

if not os.path.exists("/tmp/ComfyUI"):
    !git clone --depth 1 https://github.com/Comfy-Org/ComfyUI.git /tmp/ComfyUI

if not os.path.exists("/tmp/minimax_repo"):
    !git clone https://github.com/chinokikiss/minimax-h3-2xt4.git /tmp/minimax_repo
else:
    !cd /tmp/minimax_repo && git pull origin main

sys.path.insert(0, "/tmp/minimax_repo")
sys.path.insert(0, "/tmp/ComfyUI")
print("Environment and repo setup complete.")


In [ ]:
# Cell 3: Measure Empirical Inter-GPU PCIe Bandwidth
from benchmarks.benchmark_te_strategies import measure_pcie_bandwidth

dev0 = torch.device("cuda:0")
dev1 = torch.device("cuda:1" if num_gpus >= 2 else dev0)

pcie_stats = measure_pcie_bandwidth(dev0, dev1)
print("\nInter-GPU PCIe Bandwidth benchmark complete.")


In [ ]:
# Cell 4: Execute Text Encoder (Qwen3-VL-32B) Benchmark (PP=2 vs TP=2 vs TP=2+SP+INT8 AG)
from benchmarks.benchmark_te_strategies import benchmark_te_suite

print("\n=======================================================")
print("RUNNING TEXT ENCODER (QWEN3-VL-32B) MULTI-GPU BENCHMARK")
print("=======================================================")
te_data = benchmark_te_suite("/kaggle/working")
print("Text Encoder benchmark completed successfully.")


In [ ]:
# Cell 5: Execute DiT (Ref2VA Pruned 50 Blocks) Benchmark (PP=2 vs TP=2 vs TP=2+SP+INT8 AG)
from benchmarks.benchmark_dit_strategies import benchmark_dit_suite

print("\n====================================================")
print("RUNNING DIFFUSION DIT (50 BLOCKS) MULTI-GPU BENCHMARK")
print("====================================================")
dit_data = benchmark_dit_suite("/kaggle/working")
print("DiT benchmark completed successfully.")


In [ ]:
# Cell 6: Execute Video VAE and Audio VAE Multi-GPU Tiling Benchmark
from benchmarks.benchmark_vae_tiling import run_vae_benchmark

print("\n=======================================================")
print("RUNNING VIDEO VAE AND AUDIO VAE MULTI-GPU BENCHMARK")
print("=======================================================")
vae_data = run_vae_benchmark("/kaggle/working")
print("VAE Tile benchmark completed successfully.")


In [ ]:
# Cell 7: Execute Video VAE ViT3D INT8 ConvRot Multi-GPU Benchmark (Single vs TP=2 vs TP=2+SP+INT8 AG)
from benchmarks.benchmark_video_vae_tp import benchmark_video_vae_tp_suite

print("\n====================================================================")
print("RUNNING VIDEO VAE VIT3D INT8 CONVROT TP MULTI-GPU BENCHMARK")
print("====================================================================")
vtp_data = benchmark_video_vae_tp_suite("/kaggle/working")
print("Video VAE TP benchmark completed successfully.")


In [ ]:
# Cell 8: Verify and Summarize Generated Artifacts
import os, pandas as pd

print("\n===================== GENERATED ARTIFACTS IN /kaggle/working =====================")
files = os.listdir("/kaggle/working")
for f in sorted(files):
    sz = os.path.getsize(os.path.join("/kaggle/working", f)) / 1024
    print(f"  {f:<35} | {sz:>8.1f} KB")

if os.path.exists("/kaggle/working/te_strategies_results.csv"):
    print("\n--- TEXT ENCODER BENCHMARK RESULTS ---")
    df_te = pd.read_csv("/kaggle/working/te_strategies_results.csv")
    print(df_te.to_string(index=False))

if os.path.exists("/kaggle/working/dit_benchmark_results.csv"):
    print("\n--- DIT BENCHMARK RESULTS ---")
    df_dit = pd.read_csv("/kaggle/working/dit_benchmark_results.csv")
    print(df_dit[["seq_len", "strategy", "step_lat_ms", "turbo_8step_s", "comm_overhead_pct", "peak_vram_gb", "cos_sim"]].to_string(index=False))

if os.path.exists("/kaggle/working/video_vae_benchmark_results.csv"):
    print("\n--- VIDEO VAE TILED RESULTS ---")
    df_vvae = pd.read_csv("/kaggle/working/video_vae_benchmark_results.csv")
    print(df_vvae[["workload", "mode", "tile_size", "latency_ms", "speedup_vs_1gpu_tiled", "peak_vram_dev0_gb", "cos_sim"]].to_string(index=False))

if os.path.exists("/kaggle/working/audio_vae_benchmark_results.csv"):
    print("\n--- AUDIO VAE RESULTS ---")
    df_avae = pd.read_csv("/kaggle/working/audio_vae_benchmark_results.csv")
    print(df_avae.to_string(index=False))

if os.path.exists("/kaggle/working/video_vae_tp_benchmark_results.csv"):
    print("\n--- VIDEO VAE VIT3D INT8 TP STRATEGIES RESULTS ---")
    df_vtp = pd.read_csv("/kaggle/working/video_vae_tp_benchmark_results.csv")
    print(df_vtp[["workload", "tokens", "strategy", "decode_latency_ms", "speedup", "comm_lat_ms", "comm_volume_mb", "peak_vram_dev0_gb", "cos_sim"]].to_string(index=False))
print("==================================================================================")
